In [1]:

# STEP 1 — LOAD DATASETS

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW_DIR = Path("../data/raw")

products = pd.read_csv(RAW_DIR / "products.csv")
customers = pd.read_csv(RAW_DIR / "customers.csv")
orders = pd.read_csv(RAW_DIR / "orders.csv")
order_items = pd.read_csv(RAW_DIR / "order_items.csv")
reviews = pd.read_csv(RAW_DIR / "reviews.csv")

datasets = {
    "products": products,
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "reviews": reviews,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

print("\nRelationship key checks:")
print("Orders without customers:",
      (~orders["customer_id"].isin(customers["id"])).sum())

print("Order items without orders:",
      (~order_items["order_id"].isin(orders["id"])).sum())

print("Order items without products:",
      (~order_items["product_id"].isin(products["id"])).sum())

print("Reviews without customers:",
      (~reviews["customer_id"].isin(customers["id"])).sum())

print("Reviews without products:",
      (~reviews["product_id"].isin(products["id"])).sum())


products: (2000, 19)
customers: (800, 13)
orders: (3000, 21)
order_items: (6219, 10)
reviews: (2500, 10)

Relationship key checks:
Orders without customers: 0
Order items without orders: 0
Order items without products: 0
Reviews without customers: 0
Reviews without products: 0


In [2]:

# STEP 2 — REBUILD CUSTOMER-LEVEL METRICS

customer_order_metrics = (
    orders.groupby("customer_id")
    .agg(
        calculated_orders=("id", "count"),
        calculated_total_spent=("total", "sum"),
        calculated_delivered_orders=(
            "status",
            lambda s: s.eq("delivered").sum()
        ),
        calculated_delivered_value=(
            "total",
            lambda s: s[orders.loc[s.index, "status"].eq("delivered")].sum()
        )
    )
    .reset_index()
)

customer_analysis = customers.merge(
    customer_order_metrics,
    left_on="id",
    right_on="customer_id",
    how="left",
    validate="one_to_one"
)

metric_columns = [
    "calculated_orders",
    "calculated_total_spent",
    "calculated_delivered_orders",
    "calculated_delivered_value",
]

customer_analysis[metric_columns] = (
    customer_analysis[metric_columns].fillna(0)
)

customer_analysis["orders_count_matches"] = (
    customer_analysis["orders_count"]
    == customer_analysis["calculated_orders"]
)

customer_analysis["total_spent_matches"] = np.isclose(
    customer_analysis["total_spent"],
    customer_analysis["calculated_total_spent"],
    atol=0.01
)

display(
    customer_analysis[
        [
            "id",
            "orders_count",
            "calculated_orders",
            "total_spent",
            "calculated_total_spent",
            "orders_count_matches",
            "total_spent_matches",
        ]
    ].head(10)
)

print("\nOrder-count matches:",
      customer_analysis["orders_count_matches"].sum(), "/",
      len(customer_analysis))

print("Total-spent matches:",
      customer_analysis["total_spent_matches"].sum(), "/",
      len(customer_analysis))


,id,orders_count,calculated_orders,total_spent,calculated_total_spent,orders_count_matches,total_spent_matches
0,1,4,6.0,5419.75,8163.84,False,False
1,2,4,4.0,11795.61,11795.61,True,True
2,3,5,5.0,3138.05,3138.05,True,True
3,4,2,2.0,468.40,468.40,True,True
4,5,3,4.0,3787.59,6204.91,False,False
5,6,0,1.0,0.00,308.56,False,False
6,7,4,4.0,4967.00,4967.00,True,True
7,8,2,3.0,1765.24,2091.88,False,False
8,9,2,2.0,612.14,612.14,True,True
9,10,4,5.0,5882.65,6678.86,False,False



Order-count matches: 566 / 800
Total-spent matches: 566 / 800


### Customer-Level Metric Reconciliation

Customer-level metrics were recalculated from the orders table and compared with the existing customer records.

- The recorded order count matched the calculated order count for 566 of 800 customers (70.75%).
- Recorded total spending matched the calculated sum of order totals for the same 566 customers (70.75%).
- The remaining 234 customers had at least one metric that did not match the recalculated values.

These discrepancies indicate that the definitions or calculation rules behind the existing customer metrics may differ from the assumptions used in this analysis. The exact reason cannot be established from the available data.

**Decision:** Preserve the original customer metrics and the independently calculated metrics as separate fields. Do not overwrite or remove records solely because of these discrepancies. Use clearly defined order-status filters for subsequent customer behaviour and revenue analyses.

In [3]:

# STEP 3 — RECONCILE ORDER SUBTOTALS WITH ORDER ITEMS

item_totals = (
    order_items.groupby("order_id")
    .agg(
        calculated_subtotal=("line_total", "sum"),
        item_count=("product_id", "count"),
        total_quantity=("quantity", "sum")
    )
    .reset_index()
)

order_reconciliation = orders.merge(
    item_totals,
    left_on="id",
    right_on="order_id",
    how="left",
    validate="one_to_one"
)

order_reconciliation["calculated_subtotal"] = (
    order_reconciliation["calculated_subtotal"].fillna(0)
)

order_reconciliation["subtotal_matches"] = np.isclose(
    order_reconciliation["subtotal"],
    order_reconciliation["calculated_subtotal"],
    atol=0.01
)

print(
    "Subtotal matches:",
    order_reconciliation["subtotal_matches"].sum(),
    "/",
    len(order_reconciliation)
)

print(
    "Mismatched orders:",
    (~order_reconciliation["subtotal_matches"]).sum()
)

display(
    order_reconciliation.loc[
        ~order_reconciliation["subtotal_matches"],
        [
            "id",
            "status",
            "subtotal",
            "calculated_subtotal",
            "item_count",
            "total_quantity"
        ]
    ].head(10)
)


Subtotal matches: 3000 / 3000
Mismatched orders: 0


,id,status,subtotal,calculated_subtotal,item_count,total_quantity


### Order Subtotal Reconciliation

The subtotal of each order was compared with the sum of its associated order-item line totals.

All 3,000 orders matched within a tolerance of $0.01, with zero mismatches.

This confirms consistency between the recorded order subtotals and their associated order-item values. However, it does not independently validate other financial fields, such as discounts, taxes, shipping fees, or the final order total.

In [4]:

# STEP 4 — PRODUCT-LEVEL SALES METRICS

product_sales = (
    order_items.groupby("product_id")
    .agg(
        units_ordered=("quantity", "sum"),
        order_line_count=("order_id", "count"),
        unique_orders=("order_id", "nunique"),
        recorded_item_value=("line_total", "sum")
    )
    .reset_index()
)

product_analysis = products.merge(
    product_sales,
    left_on="id",
    right_on="product_id",
    how="left",
    validate="one_to_one"
)

sales_columns = [
    "units_ordered",
    "order_line_count",
    "unique_orders",
    "recorded_item_value",
]

product_analysis[sales_columns] = (
    product_analysis[sales_columns].fillna(0)
)

print("Products:", len(product_analysis))
print(
    "Products with recorded order items:",
    (product_analysis["order_line_count"] > 0).sum()
)
print(
    "Products without recorded order items:",
    (product_analysis["order_line_count"] == 0).sum()
)

print("\nTop 10 products by recorded item value:")

display(
    product_analysis[
        [
            "id",
            "category_id",
            "price",
            "units_ordered",
            "unique_orders",
            "recorded_item_value",
            "status",
            "in_stock",
        ]
    ]
    .sort_values("recorded_item_value", ascending=False)
    .head(10)
    .round(2)
)


Products: 2000
Products with recorded order items: 1733
Products without recorded order items: 267

Top 10 products by recorded item value:


,id,category_id,price,units_ordered,unique_orders,recorded_item_value,status,in_stock
1548,1549,2,2808.22,13.0,9.0,36506.86,active,False
1421,1422,2,2842.61,12.0,5.0,34111.32,active,True
1329,1330,2,2572.97,13.0,6.0,33448.61,active,True
1114,1115,2,2984.71,11.0,8.0,32831.81,active,False
1256,1257,2,3492.50,9.0,4.0,31432.50,active,True
677,678,2,2670.30,11.0,7.0,29373.30,active,True
1186,1187,2,1720.76,16.0,9.0,27532.16,active,True
765,766,2,3038.41,9.0,3.0,27345.69,active,True
709,710,2,2464.38,11.0,7.0,27108.18,active,False
1575,1576,2,3263.59,8.0,5.0,26108.72,active,True


### Product-Level Sales Findings

The ten products with the highest recorded item values all belong to Category 2. Product 1549 ranked first, with a recorded item value of $36,506.86 across 13 units and 9 unique orders. Product 1187 had the highest unit volume among these ten products, with 16 units across 9 unique orders.

Three of the ten products were marked as out of stock despite having recorded order-item entries. This is not necessarily contradictory, as the current inventory status may differ from the status at the time of ordering.

These figures represent aggregated order-item values across all recorded order statuses. They should not be interpreted as confirmed realized revenue.

In [5]:
category_sales = (
    product_analysis.groupby("category_id")
    .agg(
        product_count=("id", "nunique"),
        products_with_orders=("order_line_count", lambda s: (s > 0).sum()),
        total_units_ordered=("units_ordered", "sum"),
        total_unique_product_orders=("unique_orders", "sum"),
        recorded_item_value=("recorded_item_value", "sum"),
    )
    .reset_index()
    .sort_values("recorded_item_value", ascending=False)
)

display(category_sales.round(2))

,category_id,product_count,products_with_orders,total_units_ordered,total_unique_product_orders,recorded_item_value
1,2,260,220,1066.0,745.0,1858830.21
4,5,261,227,1166.0,803.0,688252.21
0,1,244,200,996.0,707.0,436129.87
3,4,246,209,1058.0,760.0,424709.47
7,8,253,227,1147.0,804.0,255640.94
2,3,260,228,1204.0,820.0,214628.30
6,7,225,198,1003.0,718.0,99550.50
5,6,251,224,1223.0,862.0,61657.27


### Category-Level Sales Analysis

Category 2 recorded the highest item value at $1,858,830.21 from 1,066 units, followed by Category 5 at $688,252.21 from 1,166 units. In contrast, Category 6 had the highest unit volume, with 1,223 units, but the lowest recorded item value at $61,657.27.

These results indicate that higher unit volume does not necessarily correspond to higher sales value. Differences in product prices likely contribute to the variation across categories.

Category 8 had 227 products with recorded order items out of 253 catalogue products. Products without order-item records were retained in the analysis.

All values represent recorded order-item totals across all order statuses and should not be interpreted as confirmed realized revenue.

In [6]:
customer_order_item_analysis = (
    orders.merge(
        order_items.groupby("order_id")
        .agg(
            total_units=("quantity", "sum"),
            item_value=("line_total", "sum"),
        )
        .reset_index(),
        left_on="id",
        right_on="order_id",
        how="left",
        validate="one_to_one",
    )
    .groupby("customer_id")
    .agg(
        order_count=("id", "nunique"),
        total_units_ordered=("total_units", "sum"),
        recorded_order_value=("total", "sum"),
        recorded_item_value=("item_value", "sum"),
    )
    .reset_index()
)

customer_order_item_analysis = customers[
    ["id", "orders_count", "total_spent", "tier"]
].merge(
    customer_order_item_analysis,
    left_on="id",
    right_on="customer_id",
    how="left",
    validate="one_to_one",
)

customer_order_item_analysis[
    [
        "order_count",
        "total_units_ordered",
        "recorded_order_value",
        "recorded_item_value",
    ]
] = customer_order_item_analysis[
    [
        "order_count",
        "total_units_ordered",
        "recorded_order_value",
        "recorded_item_value",
    ]
].fillna(0)

display(
    customer_order_item_analysis.sort_values(
        "recorded_order_value", ascending=False
    ).head(10).round(2)
)

,id,orders_count,total_spent,tier,customer_id,order_count,total_units_ordered,recorded_order_value,recorded_item_value
393,394,8,30063.98,silver,394.0,9.0,31.0,33541.09,31617.06
339,340,5,22855.44,silver,340.0,6.0,24.0,24210.14,22983.58
222,223,5,23843.61,silver,223.0,5.0,22.0,23843.61,22252.94
661,662,9,23511.15,silver,662.0,9.0,30.0,23511.15,22231.52
23,24,5,17667.64,standard,24.0,7.0,25.0,21164.69,19710.87
116,117,6,18380.26,standard,117.0,7.0,31.0,20937.37,19503.21
646,647,7,20635.04,gold,647.0,8.0,33.0,20841.43,19427.56
76,77,2,20165.79,standard,77.0,2.0,12.0,20165.79,19353.60
385,386,7,20087.08,standard,386.0,7.0,23.0,20087.08,18719.75
111,112,6,18513.76,platinum,112.0,7.0,17.0,19690.94,18732.65


### Customer-Level Order and Item Analysis

Customer-level order metrics were joined with aggregated order-item data to examine order frequency, purchased quantities, and recorded values.

Customer 394 had the highest recorded order value among the ten customers displayed, with $33,541.09 across nine orders and 31 units. However, the customer table records eight orders and total spending of $30,063.98, illustrating the previously identified discrepancy between stored customer metrics and values recalculated from the orders table.

Customer 223 provides an example where the stored order count and spending match the recalculated figures. These differences reinforce the need to preserve the original customer metrics and use explicitly defined calculations for further analysis.

Recorded order values and item values are different measures. Order totals may include shipping, taxes, and discounts, whereas item values represent the sum of recorded order-line totals. Neither measure should automatically be interpreted as realized revenue across all order statuses.

In [8]:
print("Original orders:", len(orders))
print("Order-item rows:", len(order_items))

order_item_join = orders.merge(
    order_items,
    left_on="id",
    right_on="order_id",
    how="left",
    validate="one_to_many",
    suffixes=("_order", "_item"),
)

print("Rows after join:", len(order_item_join))
print(
    "Unique orders after join:",
    order_item_join["id_order"].nunique()
)
print(
    "Unique order-item records:",
    order_item_join["id_item"].nunique()
)

print("\nExpected order-item rows:", len(order_items))
print("Actual joined rows:", len(order_item_join))

assert order_item_join["id_order"].nunique() == len(orders)
assert order_item_join["id_item"].nunique() == len(order_items)
assert len(order_item_join) == len(order_items)

print("\nAll join validation checks passed!")

Original orders: 3000
Order-item rows: 6219
Rows after join: 6219
Unique orders after join: 3000
Unique order-item records: 6219

Expected order-item rows: 6219
Actual joined rows: 6219

All join validation checks passed!


### Join Cardinality Validation

The orders table was joined with the order_items table using a one-to-many relationship. The resulting dataset contained 6,219 rows, matching the original number of order-item records, while preserving all 3,000 unique orders.

The validation checks confirmed that all 6,219 order-item records remained unique after the join and that no unexpected row multiplication occurred. This supports the reliability of subsequent order-level and item-level aggregations, provided that each analysis uses the appropriate level of granularity.

In [9]:
order_totals_before_join = orders["total"].sum()

order_totals_after_join = (
    order_item_join.groupby("id_order")["total"]
    .first()
    .sum()
)

joined_total_naive = order_item_join["total"].sum()

print(f"Original order total sum: ${order_totals_before_join:,.2f}")
print(f"Recalculated once per order: ${order_totals_after_join:,.2f}")
print(f"Naive sum after join: ${joined_total_naive:,.2f}")

print(
    "\nCorrect aggregation preserved:",
    np.isclose(order_totals_before_join, order_totals_after_join)
)
print(
    "Naive joined sum equals original:",
    np.isclose(order_totals_before_join, joined_total_naive)
)

Original order total sum: $4,280,280.61
Recalculated once per order: $4,280,280.61
Naive sum after join: $11,640,903.49

Correct aggregation preserved: True
Naive joined sum equals original: False


### Join Multiplication and Double-Counting Risk

The original sum of order totals was $4,280,280.61. Grouping the joined dataset by order and retaining one total per order reproduced this value exactly.

However, directly summing the `total` column after joining orders with order_items produced $11,640,903.49. This inflated result occurs because each order total is repeated across its associated order-item rows.

This demonstrates why aggregation must match the dataset's grain. Order-level metrics should be calculated once per order, while product-level sales should be calculated from order-item records. Failing to distinguish these levels can substantially overstate financial metrics.

In [10]:
product_review_metrics = (
    reviews.groupby("product_id")
    .agg(
        calculated_review_count=("id", "count"),
        calculated_average_rating=("rating", "mean"),
    )
    .reset_index()
)

product_review_analysis = products[
    ["id", "rating", "review_count"]
].merge(
    product_review_metrics,
    left_on="id",
    right_on="product_id",
    how="left",
    validate="one_to_one",
)

product_review_analysis["calculated_review_count"] = (
    product_review_analysis["calculated_review_count"].fillna(0)
)

product_review_analysis["review_count_matches"] = (
    product_review_analysis["review_count"]
    == product_review_analysis["calculated_review_count"]
)

print(
    "Review-count matches:",
    product_review_analysis["review_count_matches"].sum(),
    "/",
    len(product_review_analysis),
)

print(
    "Products with no reviews:",
    (product_review_analysis["calculated_review_count"] == 0).sum(),
)

print("\nSample reconciliation:")
display(product_review_analysis.head(10).round(2))

Review-count matches: 2000 / 2000
Products with no reviews: 559

Sample reconciliation:


,id,rating,review_count,product_id,calculated_review_count,calculated_average_rating,review_count_matches
0,1,4.0,1,1.0,1.0,4.0,True
1,2,5.0,4,2.0,4.0,5.0,True
2,3,4.0,2,3.0,2.0,4.0,True
3,4,4.0,2,4.0,2.0,4.0,True
4,5,NaN,0,NaN,0.0,NaN,True
5,6,5.0,1,6.0,1.0,5.0,True
6,7,3.0,2,7.0,2.0,3.0,True
7,8,5.0,1,8.0,1.0,5.0,True
8,9,3.5,2,9.0,2.0,3.5,True
9,10,5.0,1,10.0,1.0,5.0,True


### Product Review Reconciliation

The product catalogue was joined with aggregated review data using a one-to-one merge after grouping reviews by product. The recorded review count matched the recalculated count for all 2,000 products.

A total of 559 products had no reviews. Their missing average ratings were retained rather than replaced with zero, because the absence of reviews does not indicate a zero-star rating.

This reconciliation confirms that the product-level review counts are consistent with the available review records.

## Final Summary

This notebook examined relationships between customers, orders, order items, products, and reviews.

Key findings:

- All tested foreign key relationships were valid.
- All 3,000 order subtotals matched the sums of their associated order-item line totals within $0.01.
- Customer-level order counts and spending did not match the stored customer metrics for every customer. The original values were retained because their intended definitions remain unclear.
- Product-level analysis showed substantial differences in recorded item value across categories. Category 2 had the highest recorded item value, while Category 6 had the highest unit volume.
- Join validation confirmed that all 3,000 orders and 6,219 order-item records were preserved.
- Summing order totals directly after the one-to-many join inflated the total from $4,280,280.61 to $11,640,903.49, demonstrating the risk of double counting.
- Product review counts matched the review records for all 2,000 products. A total of 559 products had no reviews.

**Limitations:** Recorded order totals include multiple order statuses and should not be treated as confirmed realized revenue. Customer metric definitions and the business meaning of some status and inventory fields remain uncertain.

No source data was overwritten or removed during this analysis.